# v6 - Fine-TUNE Trackastra `ctc` on our 199 pairs (Phase 2)

Zero-shot `ctc` only TIED v2 (see `docs/experiments.md` v6). This notebook FINE-TUNES the pretrained `ctc`
model on our own data to close the domain gap (zebrafish 3D fluorescence) + learn our motion/division stats.

**How Trackastra training works (from the repo):** `scripts/train.py --model <ctc-folder>` continues training
from a pretrained model. Data must be in **Cell-Tracking-Challenge (CTC) format**: per-sample folders
`<id>/01/tXXX.tif` (raw image volumes) + `<id>/01_GT/TRA/man_trackXXX.tif` (uint16 masks where each label =
a track ID, consistent across frames) + `man_track.txt` (lineage: `L B E P` = label, begin, end, parent).
Our GT is centroids-only, so we **synthesize the TRA masks** (small balls at each GT centroid, labeled by the
track derived from the GT lineage; divisions -> daughter tracks with the mother recorded as parent).

**Pipeline:** install trackastra + train deps + clone the repo (for `scripts/train.py`) -> download `ctc` and
print its config -> convert a SUBSET of our pairs to CTC -> run `train.py --model ctc --ndim 3 --distributed
False --logger none` -> save the fine-tuned model -> re-run the v6 edge-J A/B with the fine-tuned model.

**Everything heavy runs in a subprocess** (post-install numpy is 2.4.6; a fresh interpreter stays consistent).

### THIS IS A FIRST DRAFT - expect 1-2 debug iterations
I cannot run trackastra/`train.py` locally, so the CTC folder layout, the `--window`/arch args (must MATCH the
`ctc` config printed in cell 2), and disk/memory limits may need tweaking. **Start SMALL** (`N_TRAIN=6`,
`N_VAL=2`) to debug the format fast; scale up once it trains. Watch the printed `ctc` config and set `WINDOW`
to match it. Internet **ON**, GPU **ON**.

In [3]:
PREP_SRC = r'''
import os, shutil, torch
from platformdirs import user_data_dir
from trackastra.model import Trackastra
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
Trackastra.from_pretrained('ctc', device=dev)   # downloads to user_data_dir/models/ctc
src = os.path.join(user_data_dir('trackastra'), 'models', 'ctc')
dst = '/kaggle/working/ctc_model'
if not os.path.isdir(dst):
    shutil.copytree(src, dst)
print('ctc model ->', dst, '|', os.listdir(dst))
for cf in ('config.yaml', 'train_config.yaml'):
    p = os.path.join(dst, cf)
    if os.path.exists(p):
        print('====', cf, '====')
        print(open(p).read())
'''
CONVERT_SRC = r'''
import os, glob, sys, json, shutil
from collections import defaultdict
import numpy as np, zarr, tifffile

INPUT = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
OUT   = '/kaggle/working/ctc_data'
N_TRAIN, N_VAL = int(sys.argv[1]), int(sys.argv[2])
BALL = (1, 3, 3)   # (z,y,x) half-size of the synthesized GT mask ball
MIN_RUN = 8        # need a contiguous no-empty-frame run of at least this many frames (>= window+few)

if os.path.isdir(OUT): shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)

def open_image(zp):
    g = zarr.open(zp, mode='r')
    if hasattr(g, 'shape'): return g
    best, bs = None, -1
    def walk(n):
        nonlocal best, bs
        try:
            for k in n.keys():
                s = n[k]
                if hasattr(s, 'shape'):
                    z = int(np.prod(s.shape))
                    if z > bs: best, bs = s, z
                else: walk(s)
        except Exception: pass
    walk(g); return best

def load_geff(gp):
    g = zarr.open(gp, mode='r')
    ids = np.asarray(g['nodes/ids'][:])
    props = {k: np.asarray(g['nodes/props/'+k+'/values'][:]) for k in g['nodes/props'].keys()}
    try: edges = np.asarray(g['edges/ids'][:])
    except Exception: edges = np.zeros((0, 2))
    return ids, props, edges

def longest_run(present):
    if not present: return None
    ts = sorted(present); best = (ts[0], ts[0]); cur = (ts[0], ts[0])
    for t in ts[1:]:
        if t == cur[1] + 1: cur = (cur[0], t)
        else:
            if cur[1]-cur[0] > best[1]-best[0]: best = cur
            cur = (t, t)
    if cur[1]-cur[0] > best[1]-best[0]: best = cur
    return best

def convert(zp, gp, root):
    ids, props, edges = load_geff(gp)
    node = {int(ids[i]): (int(props['t'][i]), int(props['z'][i]), int(props['y'][i]), int(props['x'][i]))
            for i in range(len(ids))}
    run = longest_run(set(node[n][0] for n in node))
    if run is None: return None
    t0, t1 = run
    if t1 - t0 + 1 < MIN_RUN:
        return (0, 0, 0, t1 - t0 + 1)                        # too short -> caller skips (nothing written)
    keep = {n for n in node if t0 <= node[n][0] <= t1}
    out_map, in_map = defaultdict(list), defaultdict(list)
    for s, d in edges:
        s, d = int(s), int(d)
        if s in keep and d in keep: out_map[s].append(d); in_map[d].append(s)
    # CTC track labels on the trimmed subgraph, times remapped to 0-based
    order = sorted(keep, key=lambda n: node[n][0])
    track = {}; parent_of = {}; span = {}; nxt = 1
    for n in order:
        ins = in_map.get(n, []); p = ins[0] if len(ins) == 1 else None
        if p is not None and len(out_map.get(p, [])) == 1:
            lab = track[p]
        else:
            lab = nxt; nxt += 1
            parent_of[lab] = track[p] if p is not None else 0
        track[n] = lab; rt = node[n][0] - t0
        span[lab] = (min(span[lab][0], rt), max(span[lab][1], rt)) if lab in span else (rt, rt)
    arr = open_image(zp); Z, Y, X = arr.shape[1], arr.shape[2], arr.shape[3]
    rz, ry, rx = BALL; K = t1 - t0
    by_t = defaultdict(list)
    for n in keep: by_t[node[n][0]].append(n)
    img_dir = os.path.join(root, '01'); tra_dir = os.path.join(root, '01_GT', 'TRA')
    os.makedirs(img_dir, exist_ok=True); os.makedirs(tra_dir, exist_ok=True)
    for rt in range(K + 1):
        t = t0 + rt
        vol = np.asarray(arr[t]).astype(np.uint16)
        tifffile.imwrite(os.path.join(img_dir, 't%03d.tif' % rt), vol, compression='zlib')
        m = np.zeros((Z, Y, X), np.uint16)
        for n in by_t.get(t, []):
            _, z, y, x = node[n]; lab = track[n]
            z0, z1 = max(0, z-rz), min(Z, z+rz+1); y0, y1 = max(0, y-ry), min(Y, y+ry+1); x0, x1 = max(0, x-rx), min(X, x+rx+1)
            sub = m[z0:z1, y0:y1, x0:x1]; sub[sub == 0] = lab
            m[z, y, x] = lab
        tifffile.imwrite(os.path.join(tra_dir, 'man_track%03d.tif' % rt), m, compression='zlib')
    with open(os.path.join(tra_dir, 'man_track.txt'), 'w') as f:
        for lab in sorted(span):
            b, e = span[lab]; f.write('%d %d %d %d\n' % (lab, b, e, parent_of.get(lab, 0)))
    return (len(keep), nxt-1, sum(1 for n in keep if len(out_map.get(n, [])) >= 2), K + 1)

pairs = [(g[:-5]+'.zarr', g) for g in sorted(glob.glob(os.path.join(INPUT, '*.geff'))) if os.path.exists(g[:-5]+'.zarr')]

def pick(cands, n, tag):
    picked = []
    for zp, gp in cands:
        sid = os.path.basename(gp)[:-5]
        if sid in picked: continue
        res = convert(zp, gp, os.path.join(OUT, sid))
        if res is None or res[0] == 0:
            runlen = 0 if res is None else res[3]
            print('  [%s] SKIP %s (dense run %d < %d)' % (tag, sid, runlen, MIN_RUN)); continue
        nn, nt, nd, runlen = res
        print('  [%s] keep %s | nodes %d tracks %d divisions %d run %d' % (tag, sid, nn, nt, nd, runlen))
        picked.append(sid)
        if len(picked) >= n: break
    return picked

train = pick(pairs, N_TRAIN, 'train')                 # scan from the front (44b6 first)
val   = pick(list(reversed(pairs)), N_VAL, 'val')     # scan from the back (6bba first)
json.dump({'train': train, 'val': val}, open(os.path.join(OUT, '_split.json'), 'w'))
print('done. train', train, '| val', val)
'''
print('subprocess scripts ready')

subprocess scripts ready


In [4]:
# ---- setup: install trackastra + training deps, clone the repo for scripts/train.py, fetch `ctc` + its config ----
import subprocess, sys, os
# train.py imports wandb/git/configargparse/lightning at module load even with --logger none, so install them.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'trackastra', 'configargparse', 'gitpython',
                'lightning', 'wandb', 'psutil', 'pyyaml', 'tifffile'], check=False)
if not os.path.isdir('/kaggle/working/trackastra_repo'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/weigertlab/trackastra',
                    '/kaggle/working/trackastra_repo'], check=False)
print('train.py present:', os.path.exists('/kaggle/working/trackastra_repo/scripts/train.py'))
# download ctc + print its config (in a fresh subprocess) so we can MATCH --window/arch args below
with open('/tmp/_prep.py', 'w') as f: f.write(PREP_SRC)
r = subprocess.run([sys.executable, '/tmp/_prep.py'], capture_output=True, text=True)
print(r.stdout[-6000:]); print('ERR:', r.stderr[-2000:])

train.py present: True
/root/.local/share/trackastra/models/ctc already downloaded, skipping.
ctc model -> /kaggle/working/ctc_model | ['model.pt', 'train_config.yaml', '.DS_Store', 'config.yaml']
==== config.yaml ====
attn_dist_mode: v0
attn_positional_bias: rope
attn_positional_bias_n_spatial: 16
causal_norm: quiet_softmax
coord_dim: 3
d_model: 512
dropout: 0.0
feat_dim: 12
feat_embed_per_dim: 8
nhead: 4
num_decoder_layers: 6
num_encoder_layers: 6
pos_embed_per_dim: 32
spatial_pos_cutoff: 256
window: 4

==== train_config.yaml ====
attn_dist_mode: v0
attn_positional_bias: rope
attn_positional_bias_n_spatial: 16
augment: 3
batch_size: 16
cache: true
cachedir: data/.ctc_dec24
causal_norm: quiet_softmax
compress: true
config: /home/gallusse/code/track_transformer/scripts/configs/ctc_dec24_bf.yaml
crop_size:
- 64
- 320
- 320
d_model: 512
delta_cutoff: 1
detection_folders:
- TRA
- ERR_SEG
distributed: true
div_upweight: 3.0
downscale_spatial: 1
downscale_temporal: 1
dropout: 0.0
dry: false

In [5]:
# ================= CONFIG (start SMALL to debug the format, then scale up) =================
N_TRAIN = 6          # number of train samples to convert (each ~0.8 GB of image tif -> watch /kaggle/working disk)
N_VAL   = 2          # last-N samples held out (matches our 6bba val convention)
EPOCHS  = 30
BATCH   = 1          # 3D transformer -> keep small; raise if GPU memory allows
WINDOW  = 4          # MATCHES the ctc config (window: 4). Do NOT change unless the ctc config changes.
TRAIN_SAMPLES = 200  # windows sampled per epoch
LR      = 1e-5       # fine-tuning LR (lower than the 1e-4 train-from-scratch default)
print('config set. N_TRAIN', N_TRAIN, 'N_VAL', N_VAL, 'EPOCHS', EPOCHS, 'WINDOW', WINDOW)

config set. N_TRAIN 6 N_VAL 2 EPOCHS 30 WINDOW 4


In [6]:
# ---- convert a SUBSET of our zarr+geff pairs to CTC format (in a subprocess: consistent numpy) ----
with open('/tmp/_convert.py', 'w') as f: f.write(CONVERT_SRC)
r = subprocess.run([sys.executable, '/tmp/_convert.py', str(N_TRAIN), str(N_VAL)], capture_output=True, text=True)
print(r.stdout[-6000:]); print('ERR:', r.stderr[-3000:])
import glob
print('converted sample dirs:', len(glob.glob('/kaggle/working/ctc_data/*/01')))

train 44b6_0113de3b | nodes 52 tracks 2 divisions 0
train 44b6_0b24845f | nodes 51 tracks 2 divisions 0
train 44b6_0c582fdc | nodes 71 tracks 1 divisions 0
train 44b6_0db75fae | nodes 157 tracks 6 divisions 0
train 44b6_12dfb391 | nodes 788 tracks 17 divisions 1
train 44b6_144b256d | nodes 121 tracks 2 divisions 0
val 6bba_fc83837d | nodes 957 tracks 34 divisions 0
val 6bba_fe670320 | nodes 716 tracks 40 divisions 2
done. train samples 6 | val samples 2

ERR: 
converted sample dirs: 8


In [7]:
# ---- fine-tune: run scripts/train.py --model ctc on the converted CTC data (subprocess) ----
import glob, json
OUTD = '/kaggle/working/ctc_data'
split = json.load(open(os.path.join(OUTD, '_split.json')))
tr = [os.path.join(OUTD, sid, '01') for sid in split['train']]
va = [os.path.join(OUTD, sid, '01') for sid in split['val']]
print('train dirs', len(tr), '| val dirs', len(va))
# Architecture + training args ALL match the ctc config so from_folder loads the pretrained weights cleanly.
cmd = [sys.executable, '/kaggle/working/trackastra_repo/scripts/train.py',
       '--model', '/kaggle/working/ctc_model',
       '--input_train', *tr, '--input_val', *va,
       '--ndim', '3', '--distributed', 'False', '--logger', 'none', '--device', 'cuda',
       '--epochs', str(EPOCHS), '--batch_size', str(BATCH), '--num_workers', '2',
       '--cachedir', 'None', '--outdir', '/kaggle/working/runs', '--detection_folders', 'TRA',
       '--features', 'wrfeat', '--window', str(WINDOW), '--train_samples', str(TRAIN_SAMPLES),
       '--lr', str(LR), '--timestamp', 'False', '--name', 'ft_ctc', '--resume', 'False',
       '--d_model', '512', '--num_encoder_layers', '6', '--num_decoder_layers', '6',
       '--pos_embed_per_dim', '32', '--feat_embed_per_dim', '8',
       '--attn_positional_bias', 'rope', '--attn_positional_bias_n_spatial', '16',
       '--attn_dist_mode', 'v0', '--causal_norm', 'quiet_softmax', '--spatial_pos_cutoff', '256',
       '--dropout', '0.0', '--delta_cutoff', '1', '--div_upweight', '3', '--max_tokens', '1024']
print('CMD:', ' '.join(cmd)); print()
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-14000:]); print('--- STDERR ---'); print(r.stderr[-8000:]); print('returncode', r.returncode)
import glob as _g
print('fine-tuned model files:', _g.glob('/kaggle/working/runs/ft_ctc/**/*', recursive=True)[:20])

CMD: /usr/bin/python3 /kaggle/working/trackastra_repo/scripts/train.py --model /kaggle/working/ctc_model --input_train /kaggle/working/ctc_data/44b6_0113de3b/01 /kaggle/working/ctc_data/44b6_0b24845f/01 /kaggle/working/ctc_data/44b6_0c582fdc/01 /kaggle/working/ctc_data/44b6_0db75fae/01 /kaggle/working/ctc_data/44b6_12dfb391/01 /kaggle/working/ctc_data/44b6_144b256d/01 --input_val /kaggle/working/ctc_data/6bba_fc83837d/01 /kaggle/working/ctc_data/6bba_fe670320/01 --ndim 3 --distributed False --logger none --device cuda --epochs 30 --batch_size 1 --num_workers 2 --cachedir None --outdir /kaggle/working/runs --detection_folders TRA --features wrfeat --window 4 --train_samples 200 --lr 1e-05 --timestamp False --name ft_ctc --resume False --d_model 512 --num_encoder_layers 6 --num_decoder_layers 6 --pos_embed_per_dim 32 --feat_embed_per_dim 8 --attn_positional_bias rope --attn_positional_bias_n_spatial 16 --attn_dist_mode v0 --causal_norm quiet_softmax --spatial_pos_cutoff 256 --dropout 0.0

## Evaluate the fine-tuned model (same v6 A/B: fine-tuned linker vs v2, identical detections)

Reuses the v6 eval, but loads the **fine-tuned** model via `Trackastra.from_folder(FT_MODEL_DIR)` instead of the
pretrained `ctc`. Read: **fine-tuned edge-J clearly > v2's 0.8594** (and dense samples recover) -> fine-tuning
worked, worth an LB slot + offline packaging; ~tie or worse -> fine-tuning didn't transfer on this subset
(try more train samples / epochs, or divisions).

In [ ]:
%%writefile /tmp/v6ft_eval.py

import os, glob, sys, time
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max
import zarr

# ---------------- config (detection identical to v2; only linking differs) ----------------
VOXEL   = np.array([1.625, 0.40625, 0.40625])
NORM_PCT = (50.0, 99.8)
HM_THR   = 0.3
MIN_DIST = 3
REFINE_WIN = (1, 3, 3)
MATCH_UM = 7.0
TIGHT_UM = 6.0; LOOSE_UM = 8.0; VEL_BLEND = 0.5
MAX_GAP = 1; GAP_DIST_UM = 6.0; FILTER_MIN_LEN = 4
N_VAL    = 5
BASE     = 16
STRIDES  = ((1,2,2),(1,2,2),(2,2,2),(2,2,2))
MASK_R   = (1, 2, 2)          # pseudo-mask half-size (z,y,x) voxels around each detection
CTC_MODE = 'greedy_nodiv'   # Fix A: no-division mode -> strict 1:1; removes the FP-division flood + LB landmine
INPUT    = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TRAIN_DIR = os.path.join(INPUT, 'train')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', device, '| numpy', np.__version__, '| torch', torch.__version__)

# ---------------- IO ----------------
def open_image(zp):
    g = zarr.open(zp, mode='r')
    if hasattr(g, 'shape'): return g
    best, bs = None, -1
    def walk(n):
        nonlocal best, bs
        try:
            for k in n.keys():
                s = n[k]
                if hasattr(s, 'shape'):
                    z = int(np.prod(s.shape))
                    if z > bs: best, bs = s, z
                else: walk(s)
        except Exception: pass
    walk(g); return best

def load_geff(gp):
    g = zarr.open(gp, mode='r')
    ids = np.asarray(g['nodes/ids'][:])
    props = {k: np.asarray(g['nodes/props/'+k+'/values'][:]) for k in g['nodes/props'].keys()}
    try: edges = np.asarray(g['edges/ids'][:])
    except Exception: edges = np.zeros((0,2))
    return ids, props, edges

def gt_nodes_edges(gp):
    ids, props, edges = load_geff(gp)
    nodes = [(int(ids[i]), int(props['t'][i]), int(props['z'][i]), int(props['y'][i]), int(props['x'][i]))
             for i in range(len(ids))]
    return nodes, [(int(s), int(t)) for s, t in edges]

def sample_list(d):
    out = []
    for gp in sorted(glob.glob(os.path.join(d, '*.geff'))):
        zp = gp[:-5] + '.zarr'
        if os.path.exists(zp): out.append((zp, gp))
    return out

# ---------------- detector (v1 arch, BASE16 InstanceNorm, anisotropic) ----------------
class ConvBlock(nn.Module):
    def __init__(s, ci, co):
        super().__init__()
        s.net = nn.Sequential(
            nn.Conv3d(ci, co, 3, padding=1, bias=False), nn.InstanceNorm3d(co, affine=True), nn.LeakyReLU(0.01, True),
            nn.Conv3d(co, co, 3, padding=1, bias=False), nn.InstanceNorm3d(co, affine=True), nn.LeakyReLU(0.01, True))
    def forward(s, x): return s.net(x)

class UNet3D(nn.Module):
    def __init__(s, in_ch=1, base=BASE, strides=STRIDES):
        super().__init__()
        n = len(strides); chs = [base*(2**i) for i in range(n+1)]
        s.enc, s.downs = nn.ModuleList(), nn.ModuleList(); cin = in_ch
        for i in range(n):
            s.enc.append(ConvBlock(cin, chs[i])); s.downs.append(nn.Conv3d(chs[i], chs[i], strides[i], strides[i])); cin = chs[i]
        s.bottleneck = ConvBlock(chs[n-1], chs[n])
        s.ups, s.dec = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(n)):
            s.ups.append(nn.ConvTranspose3d(chs[i+1], chs[i], strides[i], strides[i])); s.dec.append(ConvBlock(chs[i]*2, chs[i]))
        s.head = nn.Conv3d(chs[0], 1, 1)
    def forward(s, x):
        sk = []
        for e, d in zip(s.enc, s.downs): x = e(x); sk.append(x); x = d(x)
        x = s.bottleneck(x)
        for u, dec, k in zip(s.ups, s.dec, reversed(sk)):
            x = u(x)
            if x.shape[2:] != k.shape[2:]: x = F.interpolate(x, size=k.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, k], 1))
        return s.head(x)

def _norm(vol):
    v = vol.astype(np.float32); lo, hi = np.percentile(v, NORM_PCT)
    return np.clip((v-lo)/(hi-lo+1e-6), 0, 1).astype(np.float32)

@torch.no_grad()
def predict_hm(model, vol):
    v = _norm(vol); x = torch.from_numpy(np.ascontiguousarray(v)[None, None]).float().to(device)
    with torch.amp.autocast(device, enabled=(device == 'cuda')): y = torch.sigmoid(model(x))
    return y[0, 0].float().cpu().numpy()

def refine(vol, coords, win=REFINE_WIN):
    if len(coords) == 0: return coords
    Z, Y, X = vol.shape; out = coords.copy().astype(np.float64); wz, wy, wx = win
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z-wz), min(Z, z+wz+1); y0, y1 = max(0, y-wy), min(Y, y+wy+1); x0, x1 = max(0, x-wx), min(X, x+wx+1)
        p = vol[z0:z1, y0:y1, x0:x1].astype(np.float64); ss = p.sum()
        if ss <= 0: continue
        zz = np.arange(z0, z1)[:, None, None]; yy = np.arange(y0, y1)[None, :, None]; xx = np.arange(x0, x1)[None, None, :]
        out[i, 0] = (p*zz).sum()/ss; out[i, 1] = (p*yy).sum()/ss; out[i, 2] = (p*xx).sum()/ss
    return out

def detect(model, arr):
    frames = []
    for t in range(arr.shape[0]):
        vol = np.asarray(arr[t]).astype(np.float32); hm = predict_hm(model, vol)
        pk = peak_local_max(hm, min_distance=MIN_DIST, threshold_abs=HM_THR, exclude_border=False, num_peaks=200000)
        frames.append(refine(vol, pk.astype(np.float64)) if len(pk) else np.zeros((0, 3)))
    return frames

# ---------------- scorer (identical to repo edge_jaccard) ----------------
def match_nodes(gt_nodes, pred_nodes, max_um=MATCH_UM):
    gt_by, pr_by = defaultdict(list), defaultdict(list)
    for n in gt_nodes: gt_by[n[1]].append(n)
    for n in pred_nodes: pr_by[n[1]].append(n)
    g2p = {}
    for t, G in gt_by.items():
        P = pr_by.get(t, [])
        if not P: continue
        A = np.array([[g[2], g[3], g[4]] for g in G])*VOXEL
        B = np.array([[p[2], p[3], p[4]] for p in P])*VOXEL
        D = cdist(A, B); r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= max_um: g2p[G[i][0]] = P[j][0]
    return g2p

def edge_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges):
    g2p = match_nodes(gt_nodes, pred_nodes); p2g = {v: k for k, v in g2p.items()}
    pred_set = set(map(tuple, pred_edges)); gt_set = set(map(tuple, gt_edges))
    TP = FP = 0
    for a, b in pred_edges:
        if a in p2g and b in p2g:
            TP += (p2g[a], p2g[b]) in gt_set; FP += (p2g[a], p2g[b]) not in gt_set
    FN = sum(1 for u, v in gt_edges if not (u in g2p and v in g2p and (g2p[u], g2p[v]) in pred_set))
    d = TP+FP+FN
    return dict(J=TP/d if d else 0.0, TP=TP, FP=FP, FN=FN), g2p

def division_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges, g2p=None):
    if g2p is None: g2p = match_nodes(gt_nodes, pred_nodes)
    p2g = {v: k for k, v in g2p.items()}
    def outdeg(edges):
        d = defaultdict(int)
        for a, b in edges: d[a] += 1
        return d
    gd, pd = outdeg(gt_edges), outdeg(pred_edges)
    gt_div = set(u for u, c in gd.items() if c >= 2)
    pred_div = set(u for u, c in pd.items() if c >= 2)
    TP = sum(1 for u in gt_div if u in g2p and g2p[u] in pred_div)
    FN = len(gt_div) - TP
    FP = sum(1 for p in pred_div if (p2g.get(p) is None or p2g.get(p) not in gt_div))
    d = TP+FP+FN
    return dict(divJ=TP/d if d else 0.0, TP=TP, FP=FP, FN=FN, n_gt_div=len(gt_div), n_pred_div=len(pred_div))

# ---------------- v2 two-pass linking (baseline) ----------------
def build_nodes(frames):
    nodes = []; fid = []; nid = 1
    for t, coords in enumerate(frames):
        ids = []
        for (z, y, x) in coords:
            nodes.append((nid, t, z, y, x)); ids.append(nid); nid += 1
        fid.append(ids)
    return nodes, fid

def link_twopass(frames, fid, tight_um=TIGHT_UM, loose_um=LOOSE_UM, vel_blend=VEL_BLEND):
    def hun(P, C, pred, pi, ci, gate):
        if len(pi) == 0 or len(ci) == 0: return []
        BIG = 1e9
        Draw = np.sqrt(((P[pi][:, None]-C[ci][None])**2).sum(2))
        D = np.sqrt(((pred[pi][:, None]-C[ci][None])**2).sum(2))
        cost = np.where(Draw > gate, BIG, D)
        ri, rc = linear_sum_assignment(cost)
        return [(int(pi[r]), int(ci[c])) for r, c in zip(ri, rc) if cost[r, c] < BIG]
    edges = []; prev_xyz = np.zeros((0, 3)); prev_ids = []; prev_vel = None
    for t in range(len(frames)):
        curr = np.asarray(frames[t], float).reshape(-1, 3); curr_ids = fid[t]
        if t > 0 and len(prev_ids) and len(curr):
            P = prev_xyz*VOXEL[None, :]; C = curr*VOXEL[None, :]
            pred = P + (vel_blend*prev_vel if prev_vel is not None else 0.0)
            N, M = len(P), len(C)
            links = hun(P, C, pred, np.arange(N), np.arange(M), min(tight_um, loose_um))
            up = {p for p, _ in links}; uc = {c for _, c in links}
            fp = np.array([i for i in range(N) if i not in up], int)
            fc = np.array([j for j in range(M) if j not in uc], int)
            links += hun(P, C, pred, fp, fc, loose_um)
            vel = np.zeros((N, 3)); nv = np.zeros((M, 3))
            for p, c in links:
                edges.append((prev_ids[p], curr_ids[c])); vel[p] = (curr[c]-prev_xyz[p])*VOXEL
            for p, c in links: nv[c] = vel[p]
            prev_vel = nv
        else:
            prev_vel = None
        prev_xyz, prev_ids = curr, curr_ids
    return edges

def close_gaps(frames, fid, nodes, edges, max_gap=MAX_GAP, gap_dist_um=GAP_DIST_UM):
    if not edges: return nodes, edges
    coords = {n[0]: (n[1], n[2], n[3], n[4]) for n in nodes}
    has_out = set(s for s, _ in edges); has_in = set(t for _, t in edges)
    ends, starts = {}, {}
    for nid, (t, z, y, x) in coords.items():
        if nid not in has_out: ends.setdefault(t, []).append(nid)
        if nid not in has_in: starts.setdefault(t, []).append(nid)
    new_nodes = list(nodes); new_edges = list(edges); nxt = max(coords) + 1
    for gap in range(1, max_gap+1):
        for t, es in ends.items():
            ss = starts.get(t+gap+1)
            if not ss: continue
            ec = np.array([[coords[e][1], coords[e][2], coords[e][3]] for e in es])*VOXEL
            sc = np.array([[coords[s][1], coords[s][2], coords[s][3]] for s in ss])*VOXEL
            d = np.sqrt(((ec[:, None, :]-sc[None, :, :])**2).sum(2)); thr = gap_dist_um*(gap+1); big = thr*1000+1
            cost = np.where(d <= thr, d, big); ri, ci = linear_sum_assignment(cost); used = set()
            for r, c in zip(ri, ci):
                if d[r, c] > thr or es[r] in has_out or ss[c] in used: continue
                e_id, s_id = es[r], ss[c]; te, ze, ye, xe = coords[e_id]; ts, zs, yy2, xs = coords[s_id]; prev = e_id
                for k in range(1, gap+1):
                    fr = k/(gap+1); nid = nxt; nxt += 1
                    new_nodes.append((nid, te+k, ze+(zs-ze)*fr, ye+(yy2-ye)*fr, xe+(xs-xe)*fr))
                    new_edges.append((prev, nid)); prev = nid
                new_edges.append((prev, s_id)); has_out.add(e_id); used.add(c)
    return new_nodes, new_edges

def filter_short(nodes, edges, min_len=FILTER_MIN_LEN):
    if not edges or min_len <= 1: return nodes, edges
    par = {n[0]: n[0] for n in nodes}
    def find(a):
        while par[a] != a: par[a] = par[par[a]]; a = par[a]
        return a
    for s, t in edges:
        ra, rb = find(s), find(t)
        if ra != rb: par[ra] = rb
    comp = defaultdict(list)
    for n in nodes: comp[find(n[0])].append(n[0])
    keep = set()
    for m in comp.values():
        if len(m) >= min_len: keep.update(m)
    nn = [n for n in nodes if n[0] in keep]
    ee = [(s, t) for s, t in edges if s in keep and t in keep]
    return nn, ee

def v2_link(frames):
    nodes, fid = build_nodes(frames)
    edges = link_twopass(frames, fid)
    nodes, edges = close_gaps(frames, fid, nodes, edges)
    # prune isolated
    used = set();
    for s, t in edges: used.add(s); used.add(t)
    nodes = [n for n in nodes if n[0] in used]
    nodes, edges = filter_short(nodes, edges)
    pred_nodes = [(n[0], n[1], int(round(n[2])), int(round(n[3])), int(round(n[4]))) for n in nodes]
    return pred_nodes, edges

# ---------------- trackastra (ctc) linking ----------------
def make_masks(frames, shape, r=MASK_R):
    # Fix B: two passes per frame. Pass 1 paints small balls (paint-if-empty) for feature richness; pass 2
    # FORCES each detection's center voxel to its own label so NO detection is lost to ball collision
    # (the v6 run lost ~1-1.6% of detections this way, hurting the dense samples most).
    T = len(frames); masks = np.zeros((T,)+shape, np.uint16); rz, ry, rx = r
    id_map = {}; nodes = []; gid = 1
    for t, coords in enumerate(frames):
        centers = []
        for li, (z, y, x) in enumerate(coords):
            lab = li+1; zc, yc, xc = int(round(z)), int(round(y)), int(round(x))
            z0, z1 = max(0, zc-rz), min(shape[0], zc+rz+1); y0, y1 = max(0, yc-ry), min(shape[1], yc+ry+1); x0, x1 = max(0, xc-rx), min(shape[2], xc+rx+1)
            sub = masks[t, z0:z1, y0:y1, x0:x1]; sub[sub == 0] = lab   # pass 1: paint-if-empty ball
            centers.append((zc, yc, xc, lab))
            id_map[(t, lab)] = gid; nodes.append((gid, t, zc, yc, xc)); gid += 1
        for zc, yc, xc, lab in centers:
            masks[t, zc, yc, xc] = lab                                 # pass 2: force center -> every detection keeps >=1 labeled voxel
    return masks, id_map, nodes

def ctc_link(tra, arr, frames):
    T = arr.shape[0]; Z, Y, X = arr.shape[1], arr.shape[2], arr.shape[3]
    masks, id_map, nodes = make_masks(frames, (Z, Y, X))
    imgs = np.stack([_norm(np.asarray(arr[t]).astype(np.float32)) for t in range(T)]).astype(np.float32)
    present = int((np.array([masks[t].max() for t in range(T)]) > 0).sum())
    tg = tra.track(imgs, masks, mode=CTC_MODE)
    G = tg
    if not hasattr(G, 'nodes'):
        G = getattr(tg, 'graph', None)
        if G is None and isinstance(tg, (tuple, list)): G = tg[0]
    print('   tra graph type', type(tg), '| nodes', G.number_of_nodes(), 'edges', G.number_of_edges())
    sample = list(G.nodes(data=True))[:3]
    print('   sample tra nodes:', sample)
    def tl(data):
        t = data.get('time', data.get('t', data.get('frame')))
        lab = data.get('label', data.get('mask_id', data.get('seg_label', data.get('id'))))
        return t, lab
    tra2our = {}
    for n, data in G.nodes(data=True):
        t, lab = tl(data)
        if t is None or lab is None: continue
        key = (int(t), int(lab))
        if key in id_map: tra2our[n] = id_map[key]
    pred_edges = [(tra2our[u], tra2our[v]) for u, v in G.edges() if u in tra2our and v in tra2our]
    print('   mapped', len(tra2our), '/', G.number_of_nodes(), 'nodes;', len(pred_edges), 'edges; masks-present', present)
    return nodes, pred_edges

# ---------------- main ----------------
wpath = None
for pat in ('v1_UNet_best.pt', 'unet_heatmap.pt', 'unet_latest.pt'):
    hits = [h for h in sorted(glob.glob('/kaggle/input/**/'+pat, recursive=True)) if 'trackastra' not in h.lower()]
    if hits: wpath = hits[0]; break
assert wpath, 'no v1 weights found under /kaggle/input (attach biohub-v1-unet-base16-heatmap)'
print('weights:', wpath)
model = UNet3D().to(device)
ck = torch.load(wpath, map_location=device)
model.load_state_dict(ck.get('best_model') or ck.get('model') or ck if isinstance(ck, dict) else ck)
model.eval()

from trackastra.model import Trackastra
tra = Trackastra.from_folder(os.environ.get('FT_MODEL_DIR', '/kaggle/working/runs/ft_ctc'), device=device)
print('loaded FINE-TUNED model from', os.environ.get('FT_MODEL_DIR', '/kaggle/working/runs/ft_ctc'))

samples = sample_list(TRAIN_DIR)[-N_VAL:]
print('eval samples:', [os.path.basename(g)[:-5] for _, g in samples])

agg = {'v2': [0, 0, 0], 'ctc': [0, 0, 0]}  # TP,FP,FN
div_tot = {'TP': 0, 'FP': 0, 'FN': 0}
for zp, gp in samples:
    t0 = time.time(); name = os.path.basename(gp)[:-5]
    arr = open_image(zp)
    gt_nodes, gt_edges = gt_nodes_edges(gp)
    frames = detect(model, arr)
    npred = sum(len(f) for f in frames)
    # v2 baseline
    v2n, v2e = v2_link(frames)
    rv2, _ = edge_jaccard(gt_nodes, gt_edges, v2n, v2e)
    # ctc
    try:
        ctn, cte = ctc_link(tra, arr, frames)
        rct, g2p = edge_jaccard(gt_nodes, gt_edges, ctn, cte)
        rdiv = division_jaccard(gt_nodes, gt_edges, ctn, cte, g2p)
    except Exception as e:
        print('   ctc FAILED on', name, ':', repr(e)[:300]); rct = dict(J=0, TP=0, FP=0, FN=len(gt_edges)); rdiv = dict(divJ=0, TP=0, FP=0, FN=0, n_gt_div=0, n_pred_div=0)
    for k, r in (('v2', rv2), ('ctc', rct)):
        agg[k][0] += r['TP']; agg[k][1] += r['FP']; agg[k][2] += r['FN']
    div_tot['TP'] += rdiv['TP']; div_tot['FP'] += rdiv['FP']; div_tot['FN'] += rdiv['FN']
    print(f"{name}: preds={npred} | v2 J={rv2['J']:.3f} (TP{rv2['TP']} FP{rv2['FP']} FN{rv2['FN']}) | "
          f"ctc J={rct['J']:.3f} (TP{rct['TP']} FP{rct['FP']} FN{rct['FN']}) | "
          f"div TP{rdiv['TP']} FP{rdiv['FP']} FN{rdiv['FN']} ({time.time()-t0:.0f}s)")

def micro(a):
    TP, FP, FN = a; d = TP+FP+FN; return TP/d if d else 0.0
print()
print('==== MICRO edge-Jaccard (5 val, same detections) ====')
print(f"  v2  two-pass : {micro(agg['v2']):.4f}  (TP{agg['v2'][0]} FP{agg['v2'][1]} FN{agg['v2'][2]})   [reference ~0.859]")
print(f"  ctc zero-shot: {micro(agg['ctc']):.4f}  (TP{agg['ctc'][0]} FP{agg['ctc'][1]} FN{agg['ctc'][2]})")
dd = div_tot['TP']+div_tot['FP']+div_tot['FN']
print(f"  ctc division-J: {(div_tot['TP']/dd if dd else 0.0):.4f}  (TP{div_tot['TP']} FP{div_tot['FP']} FN{div_tot['FN']})")
print()
print('READ: ctc(nodiv) edge-J >= v2 -> learned linking wins on identical detections -> Phase 2 winner.')
print('      ctc < v2 -> zero-shot domain gap; next = fine-tune ctc on our 199 pairs.')


In [ ]:
# set FT_MODEL_DIR to the fine-tuned model folder, then run the v6 A/B eval in a subprocess
import subprocess, sys, os, glob, time
cands = [d for d in glob.glob('/kaggle/working/runs/ft_ctc*') if os.path.isdir(d)]
ft = None
for d in cands:
    if any(os.path.basename(p) == 'config.yaml' for p in glob.glob(os.path.join(d, '**', 'config.yaml'), recursive=True)):
        # use the folder that directly contains config.yaml + model
        hit = glob.glob(os.path.join(d, '**', 'config.yaml'), recursive=True)[0]
        ft = os.path.dirname(hit); break
ft = ft or '/kaggle/working/runs/ft_ctc'
print('FT_MODEL_DIR =', ft)
env = dict(os.environ, FT_MODEL_DIR=ft)
t0 = time.time()
r = subprocess.run([sys.executable, '/tmp/v6ft_eval.py'], capture_output=True, text=True, env=env)
print(r.stdout[-16000:]); print('--- STDERR tail ---'); print(r.stderr[-4000:])
print('elapsed', round(time.time()-t0), 's | returncode', r.returncode)